## Bussiness case:

We’re opening a chocolate business and want to study our biggest competitor, from which we have access to their sales.

[Link for dataset](https://www.kaggle.com/datasets/arjunmehta1992/chocolate-sales-in-20222023/data)

We will look at different data to compare and prepare our go to market

We will shape the data in 3 tables: invoices, salesperson, product

We want to find out:

- Their prices
- Their discounts
- Seasonality of their sales
- Best salesperson to recruit for our company
- What country buys the most?

Viz:
- Leaderboard of sales
- Leaderboard countries

## About Dataset

This dataset captures order-level sales activity for a chocolate brand operating across multiple countries and sales channels between January 2022 and December 2023. Each row represents a single sales order, covering product details, discounting, marketing spend, and the resulting shipment volume.

The data reflects a mix of retail, online, and wholesale channels, with seasonal demand patterns (notably around Valentine's Day and the year-end holiday season) and variation across 25 salespeople, 7 countries, and 12 product lines.

As with most real-world sales exports, the data contains some inconsistencies — missing fields, duplicate records, mixed date formats, and a handful of outliers — that came through during the export/merge process and were left as-is.

Content

    Order_ID — Unique identifier for each order

    Product — Chocolate product name

    Country — Country of sale

    Channel — Sales channel (Retail / Online / Wholesale)

    Salesperson — Sales representative handling the order

    Order_Date — Date the order was placed

    Discount_Pct — Discount applied to the order (%)

    Price_per_Box — Price per box after discount (USD)

    Marketing_Spend — Marketing budget allocated around the order (USD)

    Boxes_Shipped — Number of boxes shipped for the order

    Amount — Total order value (Boxes_Shipped × Price_per_Box)


## Loading libraries

In [1]:
import pandas as pd
import numpy as np
import cleaning_chocolate as cc

%load_ext autoreload
%autoreload 2

## Loading dataframe

In [2]:
chocolate_df = pd.read_csv('Chocolate_Sales.csv')
chocolate_df

,Order_ID,Product,Country,Channel,Salesperson,Order_Date,Discount_Pct,Price_per_Box,Marketing_Spend,Boxes_Shipped,Amount
0,ORD-069833,Truffle Gift Box,Australia,Retail,Arjun Mehta,2022-12-11,3.5,13.72,202.03,71,912.31
1,ORD-090726,85% Dark Bar,Australia,Retail,Arjun Mehta,2023-03-14,9.4,3.30,55.18,84,245.91
2,ORD-042159,70% Dark Bar,Japan,Retail,Hannah Müller,2023-12-21,4.9,18.21,60.65,35,583.7
3,ORD-197166,Hazelnut Milk Bar,Germany,Retail,Arjun Mehta,2023-12-18,15.0,2.66,52.00,92,211.27
4,ORD-112162,Almond Crunch Bar,Australia,Retail,Yuki Sato,2023-08-18,4.4,2.75,187.44,214,549.69
...,...,...,...,...,...,...,...,...,...,...,...
199995,ORD-053460,70% Dark Bar,Australia,Retail,Beatriz Souza,2022-05-27,10.6,3.39,186.19,125,418.56
199996,ORD-010743,Truffle Gift Box,India,Wholesale,Arjun Mehta,2022-09-06,17.3,3.07,42.28,148,366.47
199997,ORD-049690,Salted Caramel Bar,Japan,Online,Arjun Mehta,2022-06-16,18.3,2.74,100.19,149,329.79
199998,ORD-189637,85% Dark Bar,Japan,Wholesale,Rohan Patel,2022-10-14,9.3,14.30,29.96,24,304.51


## Dataset issues:

### Missing values in the dataset:

order_date 437 rows -> drop as is 1% of the total rows

discount_pct 489 -> means no discount (add 0)

price_per_box 457 -> perhaps the values are somewhere else

marketing_spend 461 -> means no money spent (add 0)

#### Amount doesn't match price_per_box * boxes shipped. When adding the discount back it seems similar
- Fix: create a new amount column price_per_box * boxes_shipped
- Fix: rename old amount to amount_deprecated
- Fix: create a whole amount per box column (price per box+discount)

### Price per box is with price discounted. 
- Fix: Add back discount to new column to know what is the original price

### Fix box shipped when they're on minus





## Cleaning

In [ ]:
#First, let's clean the column names of the chocolate DataFrame using the `clean_column_names` function from the 
# `cleaning_chocolate` module.

cleaned_chocolate_df = cc.clean_column_names(chocolate_df)
cleaned_chocolate_df

,order_id,product,country,channel,salesperson,order_date,discount_pct,price_per_box,marketing_spend,boxes_shipped,amount
0,ORD-069833,Truffle Gift Box,Australia,Retail,Arjun Mehta,2022-12-11,3.5,13.72,202.03,71,912.31
1,ORD-090726,85% Dark Bar,Australia,Retail,Arjun Mehta,2023-03-14,9.4,3.30,55.18,84,245.91
2,ORD-042159,70% Dark Bar,Japan,Retail,Hannah Müller,2023-12-21,4.9,18.21,60.65,35,583.7
3,ORD-197166,Hazelnut Milk Bar,Germany,Retail,Arjun Mehta,2023-12-18,15.0,2.66,52.00,92,211.27
4,ORD-112162,Almond Crunch Bar,Australia,Retail,Yuki Sato,2023-08-18,4.4,2.75,187.44,214,549.69
...,...,...,...,...,...,...,...,...,...,...,...
199995,ORD-053460,70% Dark Bar,Australia,Retail,Beatriz Souza,2022-05-27,10.6,3.39,186.19,125,418.56
199996,ORD-010743,Truffle Gift Box,India,Wholesale,Arjun Mehta,2022-09-06,17.3,3.07,42.28,148,366.47
199997,ORD-049690,Salted Caramel Bar,Japan,Online,Arjun Mehta,2022-06-16,18.3,2.74,100.19,149,329.79
199998,ORD-189637,85% Dark Bar,Japan,Wholesale,Rohan Patel,2022-10-14,9.3,14.30,29.96,24,304.51


In [7]:
# Drop rows with missing values in the 'order_date' column as they are just 1% of the rows.
cleaned_chocolate_df = cc.drop_blank_order_dates(cleaned_chocolate_df)


In [10]:
# Fill missing values in the 'discount_pct' and 'marketing_spend' columns with 0.
cleaned_chocolate_df = cc.fill_missing_with_zero(cleaned_chocolate_df, 'discount_pct')
cleaned_chocolate_df = cc.fill_missing_with_zero(cleaned_chocolate_df, 'marketing_spend')
cleaned_chocolate_df

,order_id,product,country,channel,salesperson,order_date,discount_pct,price_per_box,marketing_spend,boxes_shipped,amount
0,ORD-069833,Truffle Gift Box,Australia,Retail,Arjun Mehta,2022-12-11,3.5,13.72,202.03,71,912.31
1,ORD-090726,85% Dark Bar,Australia,Retail,Arjun Mehta,2023-03-14,9.4,3.30,55.18,84,245.91
2,ORD-042159,70% Dark Bar,Japan,Retail,Hannah Müller,2023-12-21,4.9,18.21,60.65,35,583.7
3,ORD-197166,Hazelnut Milk Bar,Germany,Retail,Arjun Mehta,2023-12-18,15.0,2.66,52.00,92,211.27
4,ORD-112162,Almond Crunch Bar,Australia,Retail,Yuki Sato,2023-08-18,4.4,2.75,187.44,214,549.69
...,...,...,...,...,...,...,...,...,...,...,...
199995,ORD-053460,70% Dark Bar,Australia,Retail,Beatriz Souza,2022-05-27,10.6,3.39,186.19,125,418.56
199996,ORD-010743,Truffle Gift Box,India,Wholesale,Arjun Mehta,2022-09-06,17.3,3.07,42.28,148,366.47
199997,ORD-049690,Salted Caramel Bar,Japan,Online,Arjun Mehta,2022-06-16,18.3,2.74,100.19,149,329.79
199998,ORD-189637,85% Dark Bar,Japan,Wholesale,Rohan Patel,2022-10-14,9.3,14.30,29.96,24,304.51
